In [13]:
!lsb_release -a

No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 26.04 LTS
Release:	26.04
Codename:	resolute


In [14]:
import pandas as pd
import cupy as cp
import cudf
import cuml
import torch
import gc
import time
from cuml import KMeans
from cuml.cluster import KMeans

In [15]:
df_pandas = pd.read_csv('household_power_consumption_not_null.csv', parse_dates=[['Date', 'Time']],
                       date_format = {'Date': '%d/%m/%Y', 
                                      'Time': '%H:%M:%S'},
                       dayfirst = True)
df_pandas

/tmp/ipykernel_17906/4030125953.py:1: FutureWarning: Support for nested sequences for 'parse_dates' in pd.read_csv is deprecated. Combine the desired columns with pd.to_datetime after parsing instead.
  df_pandas = pd.read_csv('household_power_consumption_not_null.csv', parse_dates=[['Date', 'Time']],


,Date_Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0
...,...,...,...,...,...,...,...,...
2049275,2010-11-26 20:58:00,0.946,0.000,240.43,4.0,0.0,0.0,0.0
2049276,2010-11-26 20:59:00,0.944,0.000,240.00,4.0,0.0,0.0,0.0
2049277,2010-11-26 21:00:00,0.938,0.000,239.82,3.8,0.0,0.0,0.0
2049278,2010-11-26 21:01:00,0.934,0.000,239.70,3.8,0.0,0.0,0.0


In [16]:
df_cudf_1 = cudf.DataFrame(df_pandas)
df_cudf_1

,Date_Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0
...,...,...,...,...,...,...,...,...
2049275,2010-11-26 20:58:00,0.946,0.000,240.43,4.0,0.0,0.0,0.0
2049276,2010-11-26 20:59:00,0.944,0.000,240.00,4.0,0.0,0.0,0.0
2049277,2010-11-26 21:00:00,0.938,0.000,239.82,3.8,0.0,0.0,0.0
2049278,2010-11-26 21:01:00,0.934,0.000,239.70,3.8,0.0,0.0,0.0


In [17]:
df_cudf_1 = df_cudf_1.astype(float)
df_cudf_1

,Date_Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,1.166290e+18,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,1.166290e+18,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,1.166290e+18,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,1.166290e+18,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,1.166290e+18,3.666,0.528,235.68,15.8,0.0,1.0,17.0
...,...,...,...,...,...,...,...,...
2049275,1.290805e+18,0.946,0.000,240.43,4.0,0.0,0.0,0.0
2049276,1.290805e+18,0.944,0.000,240.00,4.0,0.0,0.0,0.0
2049277,1.290805e+18,0.938,0.000,239.82,3.8,0.0,0.0,0.0
2049278,1.290805e+18,0.934,0.000,239.70,3.8,0.0,0.0,0.0


In [18]:
class Clustering(object):

    def __init__(self, dataset):
        self.dataset = dataset.copy().reset_index(drop = True)

    def KMeans(self):
        global kmeans_global
        global labels_global
        global cluster_centers_global
        
        kmeans_float = KMeans(n_clusters=8, max_iter=300, tol=0.0001, verbose=False, random_state=None, init='scalable-k-means++', 
                              n_init='auto', oversampling_factor=2.0, max_samples_per_batch=32768, output_type=None)
        kmeans = kmeans_float.fit(self.dataset)
        kmeans_global = kmeans

        labels = kmeans_float.labels_
        labels_global = labels

        cluster_centers = kmeans_float.cluster_centers_
        cluster_centers_global = cluster_centers
        
        
    def main(self):
        st = time.time()
        self.KMeans()
        et = time.time()
        elapsed_time = et - st
        print('Execution time:', elapsed_time, 'seconds')

In [19]:
clust = Clustering(df_cudf_1)

In [20]:
clust.main()

Execution time: 0.46811699867248535 seconds


In [21]:
kmeans_global

KMeans()

In [22]:
labels_global

0          5
1          5
2          5
3          5
4          5
          ..
2049275    2
2049276    2
2049277    2
2049278    2
2049279    2
Length: 2049280, dtype: int32

In [23]:
cluster_centers_global

,0,1,2,3,4,5,6,7
0,1.252088e+18,0.926227,0.150903,241.421262,3.953711,0.964037,1.033109,6.164188
1,1.189668e+18,0.978868,0.115473,239.331794,4.191945,1.098189,1.575226,5.321436
2,1.282987e+18,0.935897,0.137264,241.123577,3.991993,0.929510,1.006331,6.060039
3,1.220833e+18,0.935309,0.125428,240.253931,4.003325,1.025253,1.083659,5.464461
4,1.267539e+18,1.234936,0.122080,242.342655,5.155355,1.103346,1.211704,8.629312
5,1.174013e+18,1.263407,0.121973,239.304944,5.381194,1.333615,1.709053,6.126334
6,1.205271e+18,1.248091,0.111043,240.825906,5.270961,1.253520,1.560323,6.744862
7,1.236413e+18,1.206245,0.106659,242.157159,5.056779,1.258403,1.192930,7.186678
